# 02 — Chunking and Embeddings

Experiment with chunk size / overlap, generate embeddings, and build the FAISS vector index that the Streamlit app will use at query time.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from document_loader import load_documents
from chunking import chunk_documents

DOCS_DIR = os.path.join('..', 'data', 'documents')
records = load_documents(DOCS_DIR)
print(f'{len(records)} pages loaded')


## Experiment: chunk size / overlap sweep
We try a few (chunk_size, overlap) combinations and compare resulting chunk counts and average chunk length, to justify our final choice.

In [ ]:
configs = [
    (100, 20),
    (220, 40),   # chosen default -- see rationale in chunking.py docstring
    (400, 60),
]

for size, overlap in configs:
    chunks = chunk_documents(records, chunk_size_words=size, overlap_words=overlap)
    avg_len = sum(len(c.text.split()) for c in chunks) / len(chunks)
    print(f'size={size}, overlap={overlap} -> {len(chunks)} chunks, avg {avg_len:.0f} words/chunk')


**Observation:** very small chunks (100 words) fragment a single policy clause across multiple chunks, which can split a fact (e.g., a number) away from the sentence that gives it meaning. Very large chunks (400 words) mix multiple unrelated policy sections into a single embedding, diluting the semantic signal for a specific question. 220 words with 40-word overlap keeps each chunk close to one coherent policy point while preserving enough surrounding context, and was selected as the default.

In [ ]:
# Build final chunks with the chosen configuration
chunks = chunk_documents(records, chunk_size_words=220, overlap_words=40)
print(f'Final: {len(chunks)} chunks')
chunks[0]


## Generate embeddings and build the FAISS index

Requires internet access to download `all-MiniLM-L6-v2` from Hugging Face on first run (cached locally afterward).

In [ ]:
from retriever import VectorStore

store = VectorStore()
store.build(chunks)

STORE_DIR = os.path.join('..', 'vectorstore')
store.save(STORE_DIR)
print(f'Saved index with {len(chunks)} chunks to {STORE_DIR}')


In [ ]:
# Sanity check retrieval
results = store.search('How many annual leave days are available?', top_k=3)
for r in results:
    print(f'[{r.score:.3f}] {r.doc_name} p{r.page_number}')
    print(r.text[:150], '...\n')
